# SpexAI inference walkthrough

A step-by-step fit of an X-ray spectrum with the operator emulator: load a
response, build the model, tie element abundances, run **emcee** and
**UltraNest**, and make convergence, corner, and posterior-predictive figures.

Everything here runs on CPU in a few minutes with just the model store
(`spexai/models/`) and one instrument response. For evaluation, the bias study,
and the scripts, see `docs/tutorial_inference.md`.

## 0. Setup

On macOS the OpenMP workaround **must** be set before importing torch/numpy, so
keep this as the first cell. Point `RESP` at your response directory.

In [ ]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"   # macOS OpenMP workaround (harmless elsewhere)

import numpy as np
import torch
from IPython.display import Image, display

from spexai.inference.operator_model import JointOperatorModel
from spexai.inference.response import Response
from spexai.inference.abundances import AbundanceModel
from spexai.inference.simulate import simulate_observation
from spexai.inference.fitting import Param, run_emcee, run_ultranest
from spexai.inference import fit_plots, tempdist as td
from spexai.inference.absorption import Absorption

RESP = os.path.expanduser("~/work/data/spexai/responses")   # <-- edit if needed
np.random.seed(0); torch.manual_seed(0)

## 1. Load the instrument response

`Response` parses an OGIP RMF (+ optional ARF) and exposes the incident-energy
grid the model flux is evaluated on, plus `fold()` to detector channels.

In [ ]:
response = Response(f"{RESP}/aciss_aimpt_cy28.rmf", f"{RESP}/aciss_aimpt_cy28.arf")
response

## 2. Build the emulator

`JointOperatorModel` loads the per-element operators from the model store. Use a
small element set (O, Si, Fe) so the demo is fast; pass more `elements` for a
real fit. H and He are held at solar automatically.

In [ ]:
model = JointOperatorModel(device="cpu", elements=[8, 14, 26])
model

## 3. Get a spectrum to fit

Here we **simulate** an observation from the model with a known truth (so we can
check recovery). To fit *real* data instead, build an
`spexai.inference.simulate.Observation(counts=..., response=response,
exposure=..., true_params={})` from your loaded counts.

In [ ]:
truth = {"temp": 3.0, "norm": 1e10, "velocity": 200.0, "logz": -10.0,
         "abundances": {26: 0.5, 8: 0.6, 14: 0.6}}
obs = simulate_observation(model, response, truth, exposure=1e5,
                           target_counts=2e4, instrument="Chandra ACIS-S", rng=0)
ln = float(np.log10(obs.true_params["norm"]))
print(f"{obs.total_counts:,} counts; true log_norm = {ln:.3f}")

## 4. Tie element abundances

`AbundanceModel` maps fit parameters to a `{Z: value}` dict. Here a global
metallicity `Z` scales the metals that are not otherwise set (O, Si), while iron
is freed on its own. `param_names` tells you which `Param`s to create.

(With more elements loaded you can also tie a group to a fraction of iron, e.g.
`.tie([8, 14], "alpha_Fe", ref=26)` for an [alpha/Fe] ratio; note that a global
`Z` only constrains metals that are *not* freed or tied.)

In [ ]:
ab = (AbundanceModel(model.elements)
      .global_metallicity("Z")     # scales O, Si (the metals not freed below)
      .free_element(26, "Fe"))     # iron sampled on its own
print("fit parameters from abundances:", ab.param_names)
print("example ->", ab.to_abundances({"Z": 0.6, "Fe": 0.5}))

## 5. Define the fit parameters

Each `Param(name, low, high, label, truth)` is a uniform box prior. Names must
cover the temperature, `log_norm`, `velocity`, and every `ab.param_names` entry.
`fixed` carries anything not fit (here the redshift).

In [ ]:
params = [Param("temp", 0.5, 6.0, "T [keV]", truth["temp"]),
          Param("Z", 0.1, 1.5, "Z (O, Si)", 0.6),
          Param("Fe", 0.1, 1.5, "Fe", 0.5),
          Param("velocity", 0.0, 600.0, "v [km/s]", truth["velocity"]),
          Param("log_norm", ln - 1.5, ln + 1.5, r"$\log_{10}$ norm", ln)]
fixed = {"abundances": {}, "logz": -10.0}

## 6. Run MCMC (emcee)

`run_emcee` returns an `EmceeResult` with the chain, flattened post-burn-in
`samples`, autocorrelation `tau`, and `.median`.

In [ ]:
er = run_emcee(obs, model, params, fixed, abundance_model=ab,
               nwalkers=24, nsteps=600, progress=True)
print("autocorr tau:", np.round(er.tau, 1))
print("median:", dict(zip(er.names, np.round(er.median, 3))))

## 7. Run nested sampling (UltraNest)

Same likelihood and parameters, so the posteriors are directly comparable; also
gives the Bayesian evidence `logz`.

In [ ]:
ur = run_ultranest(obs, model, params, fixed, abundance_model=ab,
                   min_num_live_points=200)
print(f"ln Z = {ur.logz:.2f} +- {ur.logzerr:.2f},  ESS = {ur.ess:.0f}")

## 8. Convergence diagnostics

Walker traces + autocorrelation time (emcee), and the nested-sampling trace with
evidence/ESS (UltraNest). Burn-in is the dotted line; orange is the truth.

In [ ]:
fit_plots.plot_emcee_trace(er, "emcee_trace.png")
fit_plots.plot_ultranest_diagnostics(ur, "ultranest_diag.png")
display(Image("emcee_trace.png"), Image("ultranest_diag.png"))

## 9. Corner plot

emcee (blue) and UltraNest (orange) posteriors overlaid, with truths in black.

In [ ]:
fit_plots.plot_corner_overlay(er, ur, "corner.png")
Image("corner.png")

## 10. Posterior predictive

Data vs. posterior-median model and draws, with standardized residuals below.
(Single-temperature models only; the DEM section shows the alternative.)

In [ ]:
fit_plots.plot_posterior_predictive(obs, model, er, ur, fixed, "ppc.png")
Image("ppc.png")

## 11. Recovery table

In [ ]:
for i, p in enumerate(params):
    q16, q50, q84 = np.percentile(er.samples[:, i], [16, 50, 84])
    print(f"{p.name:10s} truth={p.truth:8.3f}  emcee={q50:8.3f} "
          f"(-{q50-q16:.3f}/+{q84-q50:.3f})")

## 12. Fitting a temperature distribution (DEM)

Instead of a single temperature, fit a Gaussian **differential emission
measure**. Build a DEM from `tempdist`, simulate a spectrum from it, and pass it
to the sampler as `dem=`; its `param_names` (here `T_mean`, `T_sigma`) become fit
parameters. Available shapes: `gaussian_T`, `gaussian_logT`, `lognormal_T`,
`TwoGaussianDEM`, and the non-parametric `BinnedDEM`.

In [ ]:
grid = td.TempGrid(0.5, 10.0, n=48)
dem = td.gaussian_T(grid)                       # params: T_mean, T_sigma

# simulate a DEM spectrum from the model (Poisson draw)
w = dem.weights({"T_mean": 4.0, "T_sigma": 1.0})
mu = model.predict_counts_dem(dem.temp_grid, w, {26: 0.5, 8: 0.5, 14: 0.5},
                              -10.0, 1e10, 200.0, response, 1e5).squeeze(0).numpy()
scale = 2e4 / mu.sum(); ln_dem = float(np.log10(1e10 * scale))
from spexai.inference.simulate import Observation
counts = np.random.default_rng(1).poisson(mu * scale).astype(np.int64)
obs_dem = Observation(counts=counts, response=response, exposure=1e5,
                      true_params={"temp": 4.0}, instrument="DEM", expected=mu*scale)

dem_params = [Param("T_mean", 1.0, 8.0, "T mean", 4.0),
              Param("T_sigma", 0.1, 3.0, "T width", 1.0),
              Param("Z", 0.1, 1.5, "Z", 0.5),
              Param("velocity", 0.0, 400.0, "v", 200.0),
              Param("log_norm", ln_dem - 1.5, ln_dem + 1.5, "log norm", ln_dem)]
ab_dem = AbundanceModel(model.elements).global_metallicity("Z")
er_dem = run_emcee(obs_dem, model, dem_params, fixed, abundance_model=ab_dem,
                   dem=dem, nwalkers=24, nsteps=600, progress=True)
for i, p in enumerate(dem_params):
    q = np.percentile(er_dem.samples[:, i], [16, 50, 84])
    print(f"{p.name:10s} truth={p.truth:7.3f}  fit={q[1]:7.3f} (-{q[1]-q[0]:.3f}/+{q[2]-q[1]:.3f})")

## 13. Galactic absorption

Add a foreground column with an `Absorption` screen. `Absorption.default()` uses
the cached `tbabs` cross-sections (Wilms+2000) when the table
`spexai/inference/data/tbabs_sigma.npz` has been built (in a HEASoft env, via
`scripts/build_tbabs_table.py`), otherwise the dependency-free `wabs`. Simulate
absorbed data with `simulate_observation(..., absorption=absn)` and `truth["n_h"]`,
then fit with `run_emcee(..., absorption=absn)`. Fix `n_h` via `fixed["n_h"]` or
fit it by adding `Param("n_h", 1e19, 1e22, ...)`.

In [ ]:
absn = Absorption.default()             # cached tbabs if built, else wabs
print("using cross-sections:", absn.name)
T = absn.transmission(np.array([0.3, 0.5, 1.0, 2.0, 6.0]), 1.4e21).numpy()
print("transmission at N_H=1.4e21:", dict(zip([0.3,0.5,1.0,2.0,6.0], np.round(T, 3))))
# fixed = {"abundances": {}, "logz": np.log10(0.0179), "n_h": 1.4e21}
# er = run_emcee(obs, model, params, fixed, abundance_model=ab, absorption=absn)

---
That's the full loop: **load response -> build model -> tie abundances -> set
priors -> MCMC + nested sampling -> convergence, corner, posterior-predictive**,
with DEM and absorption variants. For calibration/bias testing, inject with
`SpexTruthModel` and use `scripts/bias_study.py` (see `docs/tutorial_inference.md`).